
## Overview

This notebook will show you how to create and query a table or DataFrame that you uploaded to DBFS. [DBFS](https://docs.databricks.com/user-guide/dbfs-databricks-file-system.html) is a Databricks File System that allows you to store data for querying inside of Databricks. This notebook assumes that you have a file already inside of DBFS that you would like to read from.

This notebook is written in **Python** so the default cell type is Python. However, you can use different languages by using the `%LANGUAGE` syntax. Python, Scala, SQL, and R are all supported.

In [0]:
# File location and type
file_location = "/FileStore/tables/201508_station_data-1.csv"
file_type = "csv"

# CSV options
infer_schema = "false"
first_row_is_header = "false"
delimiter = ","

# The applied options are for CSV files. For other file types, these will be ignored.
df = spark.read.format(file_type) \
  .option("inferSchema", infer_schema) \
  .option("header", 'true') \
  .option("sep", delimiter) \
  .load(file_location)

display(df)






station_id,name,lat,long,dockcount,landmark,installation
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013
7,Paseo de San Antonio,37.333798,-121.886943,15,San Jose,8/7/2013
8,San Salvador at 1st,37.330165,-121.885831,15,San Jose,8/5/2013
9,Japantown,37.348742,-121.894715,15,San Jose,8/5/2013
10,San Jose City Hall,37.337391,-121.886995,15,San Jose,8/6/2013
11,MLK Library,37.335885,-121.88566,19,San Jose,8/6/2013


In [0]:
# Create a view or table

temp_table_name = "/FileStore/tables/201508_station_data-1.csv"

df.createOrReplaceTempView("station_data")



import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, regexp_replace, regexp_extract, when, array_contains, avg, sum, countDistinct,isnan,split,expr,pandas_udf
from pyspark.sql.types import DoubleType,StringType



# Zadanie 1

##Użycie poniższych funkcji: Nulls, fill, explode, drop, regexp_replace, regexp_extract, ifnull, nullIf, replace, array_contains. 

In [0]:
 # a.	Użyj poniższe funkcje Nulls, fill, explode, drop, regexp_replace, regexp_extract, ifnull, nullIf, replace, array_contains. 

 # Nulls - sprawdzanie, czy kolumna zawiera wartości NULL


df.filter(col("lat").isNull()).show()

+----------+----+---+----+---------+--------+------------+
|station_id|name|lat|long|dockcount|landmark|installation|
+----------+----+---+----+---------+--------+------------+
+----------+----+---+----+---------+--------+------------+



In [0]:
# fill - Wypełnienie wartości NULL w danej kolumnie: 

df.fillna({"name": "Brak danych"}).show()

+----------+--------------------+---------+-----------+---------+------------+------------+
|station_id|                name|      lat|       long|dockcount|    landmark|installation|
+----------+--------------------+---------+-----------+---------+------------+------------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|    San Jose|    8/6/2013|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|    San Jose|    8/5/2013|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|    San Jose|    8/6/2013|
|         5|    Adobe on Almaden|37.331415|  -121.8932|       19|    San Jose|    8/5/2013|
|         6|    San Pedro Square|37.336721|-121.894074|       15|    San Jose|    8/7/2013|
|         7|Paseo de San Antonio|37.333798|-121.886943|       15|    San Jose|    8/7/2013|
|         8| San Salvador at 1st|37.330165|-121.885831|       15|    San Jose|    8/5/2013|
|         9|           Japantown|37.348742|-121.894715|       15|    San Jose|  

In [0]:
# explode - jeśli mamy kolumne z listami i chcemy każdą wartość w osobnym wierszu
df_exploded = df.withColumn("Exploded_Landmark", explode(split(col("landmark"), " ")))
display(df_exploded)



station_id,name,lat,long,dockcount,landmark,installation,Exploded_Landmark
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013,San
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013,Jose
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013,San
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013,Jose
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013,San
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013,Jose
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013,San
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013,Jose
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013,San
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013,Jose


In [0]:
# a.	Użyj poniższe funkcje Nulls, fill, explode, drop, regexp_replace, regexp_extract, ifnull, nullIf, replace, array_contains. 

# drop - sluzy do usuwania 

df_dropped = df.drop("Exploded_Landmark")
display(df_dropped)

station_id,name,lat,long,dockcount,landmark,installation
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013
7,Paseo de San Antonio,37.333798,-121.886943,15,San Jose,8/7/2013
8,San Salvador at 1st,37.330165,-121.885831,15,San Jose,8/5/2013
9,Japantown,37.348742,-121.894715,15,San Jose,8/5/2013
10,San Jose City Hall,37.337391,-121.886995,15,San Jose,8/6/2013
11,MLK Library,37.335885,-121.88566,19,San Jose,8/6/2013


In [0]:
# regexp_replace - zamienianie wzorcow znakow 

df_replaced = df.withColumn("landmark", regexp_replace(df["landmark"], "San Jose", "SJ"))

display(df_replaced)

station_id,name,lat,long,dockcount,landmark,installation
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,SJ,8/6/2013
3,San Jose Civic Center,37.330698,-121.888979,15,SJ,8/5/2013
4,Santa Clara at Almaden,37.333988,-121.894902,11,SJ,8/6/2013
5,Adobe on Almaden,37.331415,-121.8932,19,SJ,8/5/2013
6,San Pedro Square,37.336721,-121.894074,15,SJ,8/7/2013
7,Paseo de San Antonio,37.333798,-121.886943,15,SJ,8/7/2013
8,San Salvador at 1st,37.330165,-121.885831,15,SJ,8/5/2013
9,Japantown,37.348742,-121.894715,15,SJ,8/5/2013
10,San Jose City Hall,37.337391,-121.886995,15,SJ,8/6/2013
11,MLK Library,37.335885,-121.88566,19,SJ,8/6/2013


In [0]:
# regexp_extract - pozwala wyciągnąć fragmenty tekstu pasujące do wyrażenia regularnego.
from pyspark.sql.functions import regexp_extract, col, coalesce, lit
df_reg = df.withColumn("Extracted", regexp_extract(col("name"), r"\bS\w*\b", 0))
df = df.withColumn("name", coalesce(col("name"), lit("Unknown Station")))
df_reg.show()


+----------+--------------------+---------+-----------+---------+------------+------------+---------+
|station_id|                name|      lat|       long|dockcount|    landmark|installation|Extracted|
+----------+--------------------+---------+-----------+---------+------------+------------+---------+
|         2|San Jose Diridon ...|37.329732|-121.901782|       27|    San Jose|    8/6/2013|      San|
|         3|San Jose Civic Ce...|37.330698|-121.888979|       15|    San Jose|    8/5/2013|      San|
|         4|Santa Clara at Al...|37.333988|-121.894902|       11|    San Jose|    8/6/2013|    Santa|
|         5|    Adobe on Almaden|37.331415|  -121.8932|       19|    San Jose|    8/5/2013|         |
|         6|    San Pedro Square|37.336721|-121.894074|       15|    San Jose|    8/7/2013|      San|
|         7|Paseo de San Antonio|37.333798|-121.886943|       15|    San Jose|    8/7/2013|      San|
|         8| San Salvador at 1st|37.330165|-121.885831|       15|    San Jose|    

In [0]:
# ifnull - jesli wartość w danej kolumnie jest NULL, zamienia ją na określoną wartośc
from pyspark.sql import functions as F


df_null = df.withColumn("name", F.coalesce(F.col("name"), F.lit("Unknown Station")))
display(df_null)

station_id,name,lat,long,dockcount,landmark,installation
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013
7,Paseo de San Antonio,37.333798,-121.886943,15,San Jose,8/7/2013
8,San Salvador at 1st,37.330165,-121.885831,15,San Jose,8/5/2013
9,Japantown,37.348742,-121.894715,15,San Jose,8/5/2013
10,San Jose City Hall,37.337391,-121.886995,15,San Jose,8/6/2013
11,MLK Library,37.335885,-121.88566,19,San Jose,8/6/2013


In [0]:
#  nullIf - porównuje dwie wartości i zwraca null, jeśli te wartości są równe, w przeciwnym razie zwraca pierwszą wartość

df_nullif = df.withColumn("Landmark_NullIf", expr("nullIf(landmark, 'San Jose')"))
display(df_nullif)

station_id,name,lat,long,dockcount,landmark,installation,Landmark_NullIf
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013,null
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013,null
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013,null
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013,null
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013,null
7,Paseo de San Antonio,37.333798,-121.886943,15,San Jose,8/7/2013,null
8,San Salvador at 1st,37.330165,-121.885831,15,San Jose,8/5/2013,null
9,Japantown,37.348742,-121.894715,15,San Jose,8/5/2013,null
10,San Jose City Hall,37.337391,-121.886995,15,San Jose,8/6/2013,null
11,MLK Library,37.335885,-121.88566,19,San Jose,8/6/2013,null


In [0]:
#  replace - służy do zastępowania określonych wartości w DataFrame na inne
df_replaced = df.replace('San Jose', 'San Francisco', subset=["landmark"])
display(df_replaced)

station_id,name,lat,long,dockcount,landmark,installation
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Francisco,8/6/2013
3,San Jose Civic Center,37.330698,-121.888979,15,San Francisco,8/5/2013
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Francisco,8/6/2013
5,Adobe on Almaden,37.331415,-121.8932,19,San Francisco,8/5/2013
6,San Pedro Square,37.336721,-121.894074,15,San Francisco,8/7/2013
7,Paseo de San Antonio,37.333798,-121.886943,15,San Francisco,8/7/2013
8,San Salvador at 1st,37.330165,-121.885831,15,San Francisco,8/5/2013
9,Japantown,37.348742,-121.894715,15,San Francisco,8/5/2013
10,San Jose City Hall,37.337391,-121.886995,15,San Francisco,8/6/2013
11,MLK Library,37.335885,-121.88566,19,San Francisco,8/6/2013


In [0]:
# array_contains - służy do sprawdzania, czy w kolumnie zawierającej tablicę (array) znajduje się określona wartość

df_replaced = df_replaced.withColumn("landmark_array", split(col("landmark"), " "))
df_replaced = df_replaced.withColumn("contains_San_Francisco", array_contains(col("landmark_array"), "San Francisco"))


display(df_replaced)



station_id,name,lat,long,dockcount,landmark,installation,landmark_array,contains_San_Francisco
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Francisco,8/6/2013,"List(San, Francisco)",false
3,San Jose Civic Center,37.330698,-121.888979,15,San Francisco,8/5/2013,"List(San, Francisco)",false
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Francisco,8/6/2013,"List(San, Francisco)",false
5,Adobe on Almaden,37.331415,-121.8932,19,San Francisco,8/5/2013,"List(San, Francisco)",false
6,San Pedro Square,37.336721,-121.894074,15,San Francisco,8/7/2013,"List(San, Francisco)",false
7,Paseo de San Antonio,37.333798,-121.886943,15,San Francisco,8/7/2013,"List(San, Francisco)",false
8,San Salvador at 1st,37.330165,-121.885831,15,San Francisco,8/5/2013,"List(San, Francisco)",false
9,Japantown,37.348742,-121.894715,15,San Francisco,8/5/2013,"List(San, Francisco)",false
10,San Jose City Hall,37.337391,-121.886995,15,San Francisco,8/6/2013,"List(San, Francisco)",false
11,MLK Library,37.335885,-121.88566,19,San Francisco,8/6/2013,"List(San, Francisco)",false


## Użycie 3 funkcji agregujących 

In [0]:
from pyspark.sql.functions import avg, count, min, max

# 1 avg
df_avg_docks = df.groupBy("landmark").agg(avg("dockcount").alias("avg_dockcount"))
display(df_avg_docks)

# 2  count
df2 = df.groupBy("landmark").agg(count("landmark"). alias("name_count"))
display(df2)

# 3 - sum 
df_sum_docks = df.groupBy("landmark").agg(sum("dockcount").alias("total_dockcount"))
display(df_sum_docks)

# 4 - min, max
df_min_max_docks = df.groupBy("landmark").agg(min("dockcount").alias("min_dockcount"),max("dockcount").alias("max_dockcount"))
display(df_min_max_docks)

landmark,avg_dockcount
Palo Alto,15.0
San Francisco,19.0
San Jose,16.5
Redwood City,16.428571428571427
Mountain View,16.714285714285715


landmark,name_count
Palo Alto,5
San Francisco,35
San Jose,16
Redwood City,7
Mountain View,7


landmark,total_dockcount
Palo Alto,75.0
San Francisco,665.0
San Jose,264.0
Redwood City,115.0
Mountain View,117.0


landmark,min_dockcount,max_dockcount
Mountain View,11,23
Palo Alto,11,23
Redwood City,15,25
San Francisco,15,27
San Jose,11,27


# Zadanie 2

## Funkcje UDF 

In [0]:
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import DoubleType
import pandas as pd

# funkcja do obliczania podatku
df_test = df.withColumn("dockcount", col("dockcount").cast("int"))
@pandas_udf(DoubleType())
def calculate_tax(dockcount: pd.Series) -> pd.Series:
    dockcount = pd.to_numeric(dockcount, errors='coerce')  # Zmieniamy teksty na NaN, jeżeli nie uda się konwertować
    tax = dockcount.apply(lambda x: x * 0.05 if x > 20 else 2 if pd.notna(x) else 0)
    return tax

df_with_tax = df_test.withColumn("tax", calculate_tax(df_test["dockcount"]))
display(df_with_tax)

station_id,name,lat,long,dockcount,landmark,installation,tax
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013,1.35
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013,2.0
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013,2.0
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013,2.0
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013,2.0
7,Paseo de San Antonio,37.333798,-121.886943,15,San Jose,8/7/2013,2.0
8,San Salvador at 1st,37.330165,-121.885831,15,San Jose,8/5/2013,2.0
9,Japantown,37.348742,-121.894715,15,San Jose,8/5/2013,2.0
10,San Jose City Hall,37.337391,-121.886995,15,San Jose,8/6/2013,2.0
11,MLK Library,37.335885,-121.88566,19,San Jose,8/6/2013,2.0


In [0]:

from pyspark.sql.types import BooleanType

# funkcja która sprawdza, czy w 'landmark' znajduje się słowo 'San'
def contains_san(landmark: str) -> bool:
    if landmark and "San" in landmark:
        return True
    return False
contains_san_udf = udf(contains_san, BooleanType())
df_with_san_flag = df.withColumn("contains_san", contains_san_udf(df["landmark"]))
display(df_with_san_flag)

station_id,name,lat,long,dockcount,landmark,installation,contains_san
2,San Jose Diridon Caltrain Station,37.329732,-121.901782,27,San Jose,8/6/2013,true
3,San Jose Civic Center,37.330698,-121.888979,15,San Jose,8/5/2013,true
4,Santa Clara at Almaden,37.333988,-121.894902,11,San Jose,8/6/2013,true
5,Adobe on Almaden,37.331415,-121.8932,19,San Jose,8/5/2013,true
6,San Pedro Square,37.336721,-121.894074,15,San Jose,8/7/2013,true
7,Paseo de San Antonio,37.333798,-121.886943,15,San Jose,8/7/2013,true
8,San Salvador at 1st,37.330165,-121.885831,15,San Jose,8/5/2013,true
9,Japantown,37.348742,-121.894715,15,San Jose,8/5/2013,true
10,San Jose City Hall,37.337391,-121.886995,15,San Jose,8/6/2013,true
11,MLK Library,37.335885,-121.88566,19,San Jose,8/6/2013,true
